# Triagem medidor × outros sem rótulos (rubrica de ML)

Notebook dos experimentos do artigo. O protocolo está em `ml_README.md`,
seção 1, e foi fixado antes de qualquer experimento.

**Como executar.** As fotos do cliente não fazem parte do repositório. O
notebook lê a pasta de dados e a pasta de trabalho indicadas no `.env` da raiz
deste repositório, e só reexecuta na máquina que tem esses dados. As saídas
mostram apenas contagens, métricas e gráficos em SVG.

## 1. Configuração

In [ ]:
#Célula 2
# # Configuração: versões e impressões digitais dos artefatos de entrada.
# Nenhum caminho é exibido (decisão E15): só versões e SHA-256.
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch

from meter_reader.config import get_data_dir, get_work_dir
from training.build_manifest import MANIFEST_NAME, file_sha256
from training.build_sample import SAMPLE_NAME
from training.labels import load_labeled_photos

# Rótulos do Lab02, os mesmos da rubrica de ML (E11); o notebook roda de ml-artigo/
LABELS_PATH = Path("../lab-02/rotulos/lab2_rotulos.csv")
assert LABELS_PATH.is_file(), "Execute o notebook a partir da pasta ml-artigo/."

WORK_DIR = get_work_dir()   # valida o .env; não exibe o caminho
DATA_DIR = get_data_dir()

print(f"Python       {sys.version.split()[0]}")
print(f"PyTorch      {torch.__version__} | timm {timm.__version__}")
print(f"GPU          {torch.cuda.is_available()}")
print(f"SHA-256 manifesto: {file_sha256(WORK_DIR / MANIFEST_NAME)}")
print(f"SHA-256 amostra:   {file_sha256(WORK_DIR / SAMPLE_NAME)}")
print(f"SHA-256 rótulos:   {file_sha256(LABELS_PATH)}")

# Fotos rotuladas fora do teste (o teste segue lacrado: include_test=False)
photos = load_labeled_photos(LABELS_PATH)
random_sample = [p for p in photos if p.role != "estresse_zero"]
print(f"Rotuladas fora do teste: {len(photos)} | amostra aleatória: {len(random_sample)}")

Python       3.11.4
PyTorch      2.14.1+cu126 | timm 1.0.30
GPU          True
SHA-256 manifesto: 56467c02e64f41b6c495987e4cec4c0216dc1e38e5644ae65bceca8d2199b28a
SHA-256 amostra:   a6ced4666242f3c5832d5109c1fdc9289bbc754a693a9252971ed5259c417796
SHA-256 rótulos:   b901c05c2b5e6b89fba53adf512681a8d2507ef1aa9c2597033b6f84069c2f2e
Rotuladas fora do teste: 261 | amostra aleatória: 225


c:\Projetos_Tec\leitura_automatizada_mostradores_energia_pesquisa\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Extrator: custo e paridade

Custo por foto em CPU (4 threads, piso da E04), só o modelo, sobre 50 fotos da
amostra aleatória: DINOv2 ViT-S/14 (proposta) contra a MobileNetV3-Large, a
espinha da triagem de DL. Paridade: diferença entre as características
calculadas na GPU (onde o cache é extraído) e na CPU, sem TF32 (E33).

In [ ]:
#Célula 4
# Custo em CPU: parâmetros, operações e latência por foto (só o modelo)
import logging
import time

from PIL import Image
from torch.utils.flop_counter import FlopCounterMode

from meter_reader.preprocess import enhance, preprocess
from meter_reader.triage import TriageModel
from training.dino_features import extract_views, load_dinov2, model_input

logging.getLogger("torch.utils.flop_counter").setLevel(logging.ERROR)
THREADS = 4  # piso da E04
timing_photos = random_sample[:50]


def latency_ms(fn, items, warmup=10, runs=200):
    """p50 e p95 em milissegundos, alternando entre os itens."""
    for k in range(warmup):
        fn(items[k % len(items)])
    times = []
    for k in range(runs):
        start = time.perf_counter()
        fn(items[k % len(items)])
        times.append((time.perf_counter() - start) * 1000)
    return np.percentile(times, [50, 95])


def dino_tensor(path):
    with Image.open(path) as image:
        return model_input(enhance(image))


dino = load_dinov2()  # primeira vez: baixa os pesos oficiais
mobile = TriageModel("mobilenet_v3_large")
mobile.freeze_backbone()
mobile.eval()

extractors = {
    "DINOv2 ViT-S/14": (dino, dino_tensor, dino.parameters()),
    "MobileNetV3-Large": (mobile.embed, lambda p: preprocess(p, 224), mobile.features.parameters()),
}

previous_threads = torch.get_num_threads()
torch.set_num_threads(THREADS)
rows = {}
try:
    with torch.no_grad():
        for name, (fn, prepare, params) in extractors.items():
            tensors = [prepare(p.path).unsqueeze(0) for p in timing_photos]
            with FlopCounterMode(display=False) as counter:
                dim = fn(tensors[0]).shape[1]
            p50, p95 = latency_ms(fn, tensors)
            rows[name] = {
                "entrada (A × L)": f"{tensors[0].shape[2]} × {tensors[0].shape[3]}",
                "dimensão": dim,
                "parâmetros (M)": round(sum(t.numel() for t in params) / 1e6, 2),
                "GMAC": round(counter.get_total_flops() / 2e9, 2),  # convenção do torchvision
                "p50 (ms)": round(p50, 1),
                "p95 (ms)": round(p95, 1),
            }
finally:
    torch.set_num_threads(previous_threads)

cost = pd.DataFrame(rows).T
cost

W1020 23:12:45.306000 46604 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


,entrada (A × L),dimensão,parâmetros (M),GMAC,p50 (ms),p95 (ms)
DINOv2 ViT-S/14,294 × 224,384,22.06,7.23,77.8,139.5
MobileNetV3-Large,299 × 224,960,2.97,0.29,15.7,28.2


In [ ]:
#Célula 5
# Paridade GPU x CPU das características DINOv2, sem TF32 (E33)
assert torch.cuda.is_available(), "A paridade exige a GPU."
parity_photos = random_sample[:16]
paths = [p.path for p in parity_photos]
codes = [p.pseudonym for p in parity_photos]

cpu_features = extract_views(paths, codes, 0, dino, "cpu")
gpu_features = extract_views(paths, codes, 0, dino, "cuda")
dino.to("cpu")  # extract_views move o modelo; volta para a CPU

difference = np.abs(gpu_features - cpu_features)
relative = difference.max() / np.abs(cpu_features).max()
cosine = (gpu_features * cpu_features).sum(-1) / (
    np.linalg.norm(gpu_features, axis=-1) * np.linalg.norm(cpu_features, axis=-1))
print(f"Fotos: {len(parity_photos)}")
print(f"Diferença máxima absoluta: {difference.max():.2e} | relativa: {relative:.2e}")
print(f"Similaridade de cosseno mínima GPU x CPU: {cosine.min():.8f}")

Fotos: 16
Diferença máxima absoluta: 3.53e-05 | relativa: 3.56e-06
Similaridade de cosseno mínima GPU x CPU: 0.99999988


## 3. Características do universo do agrupamento

Todas as fotos íntegras dos três lotes de desenvolvimento, órfãs incluídas
(protocolo, seção 1.1): a foto original e 5 vistas aumentadas (seção 1.3),
extraídas uma vez na GPU, sem TF32, e guardadas na pasta de trabalho. O lote
de teste não é lido aqui.

In [ ]:
#Célula 7
# Cache das características DINOv2: foto original + 5 vistas, lotes de desenvolvimento
from training.dino_features import cache_path, load_or_extract_views
from training.unlabeled import load_development_photos

N_VIEWS = 5
CACHE_NAME = "dinov2_desenvolvimento"
cache_dir = WORK_DIR / "caracteristicas"

dev_photos = load_development_photos()
dev_codes = [p.pseudonym for p in dev_photos]

start = time.perf_counter()
dev_features = load_or_extract_views([p.path for p in dev_photos], dev_codes, CACHE_NAME,
                                     N_VIEWS, cache_dir, device="cuda")
dino.to("cpu")
elapsed_min = (time.perf_counter() - start) / 60

# Conferências
position = {code: i for i, code in enumerate(dev_codes)}
labeled_codes = {p.pseudonym for p in photos}
assert labeled_codes <= position.keys(), "Há fotos rotuladas fora do universo."
assert np.isfinite(dev_features).all(), "Características com NaN ou infinito."
parity_rows = dev_features[[position[c] for c in codes], 0]   # mesma GPU, mesmas fotos
drift = np.abs(parity_rows - gpu_features[:, 0]).max()

print(f"Fotos por lote: {pd.Series([p.lot for p in dev_photos]).value_counts().sort_index().to_dict()}")
print(f"Total: {len(dev_photos)} | forma do cache: {dev_features.shape}")
print(f"Rotuladas no universo: {len(labeled_codes)} de {len(photos)}")
print(f"Diferença para a paridade (foto original): {drift:.2e}")
print(f"Tempo: {elapsed_min:.1f} min")
print(f"SHA-256 do cache: {file_sha256(cache_path(cache_dir, CACHE_NAME, N_VIEWS))}")

Fotos por lote: {'2026-05-20': 3181, '2026-05-21': 3094, '2026-05-22': 3050}
Total: 9325 | forma do cache: (9325, 6, 384)
Rotuladas no universo: 261 de 261
Diferença para a paridade (foto original): 1.96e-05
Tempo: 0.0 min
SHA-256 do cache: 17e3777351c296597ee8b29dee0cfd9ce36177ff32affa4c2fffada4ab51ef94


## 4. Pureza dos vizinhos por extrator (H2)

Pureza dos 5 vizinhos (cosseno) de cada foto da amostra aleatória, buscados
entre as próprias 225 fotos, no colapso binário medidor × outros. Estatística
de decisão: pureza balanceada (protocolo, seção 1.7). DINOv2 vence um extrator
se o IC 95% da diferença, por bootstrap pareado e estratificado por classe,
excluir 0. A baseline é padronizada; as redes, não (seção 1.4).

In [ ]:
#Célula 9
# H2: pureza dos vizinhos, DINOv2 x MobileNetV3-Large x baseline manual
from training.evaluation import OUTROS
from training.features import load_or_extract
from training.neighbors import (balanced_purity, cosine_neighbors, neighbor_purity,
                                paired_bootstrap_purity, standardize)

K_PURITY = 5
sample_idx = np.array([i for i, p in enumerate(photos) if p.role != "estresse_zero"])
is_outros = np.array([p.scene == OUTROS for p in photos])[sample_idx].astype(int)
assert len(sample_idx) == 225 and is_outros.sum() == 17

# Mesmas 225 fotos, na mesma ordem, nos três extratores
spaces = {
    "DINOv2 ViT-S/14": dev_features[[position[photos[i].pseudonym] for i in sample_idx], 0],
    "MobileNetV3-Large": load_or_extract(photos, "mobilenet_v3_large_224", cache_dir)[sample_idx],
    "Baseline manual": standardize(load_or_extract(photos, "baseline_224", cache_dir)[sample_idx]),
}
purity = {name: neighbor_purity(cosine_neighbors(x, K_PURITY), is_outros)
          for name, x in spaces.items()}

# Nível do acaso: vizinhos sorteados entre as outras 224 fotos
n = len(is_outros)
chance_meter = (n - is_outros.sum() - 1) / (n - 1)
chance_outros = (is_outros.sum() - 1) / (n - 1)

rows = {
    name: {
        "dimensão": spaces[name].shape[1],
        "pureza medidor": round(p[is_outros == 0].mean(), 3),
        "pureza outros": round(p[is_outros == 1].mean(), 3),
        "pureza balanceada": round(balanced_purity(p, is_outros), 3),
    }
    for name, p in purity.items()
}
rows["Acaso (referência)"] = {
    "dimensão": "-", "pureza medidor": round(chance_meter, 3),
    "pureza outros": round(chance_outros, 3),
    "pureza balanceada": round((chance_meter + chance_outros) / 2, 3),
}
print(f"Fotos: {n} (medidor {n - is_outros.sum()}, outros {is_outros.sum()}) | k = {K_PURITY}\n")
print(pd.DataFrame(rows).T.to_string())

# Decisão da H2: DINOv2 contra cada um dos outros extratores
print("\nDiferença da pureza balanceada (DINOv2 - outro), IC 95%:")
for other in ("MobileNetV3-Large", "Baseline manual"):
    point, low, high = paired_bootstrap_purity(purity["DINOv2 ViT-S/14"], purity[other], is_outros)
    verdict = "DINOv2 vence" if low > 0 else ("DINOv2 perde" if high < 0 else "empate")
    print(f"  x {other:18} {point:+.3f}  [{low:+.3f}; {high:+.3f}]  -> {verdict}")

Fotos: 225 (medidor 208, outros 17) | k = 5

                   dimensão pureza medidor pureza outros pureza balanceada
DINOv2 ViT-S/14       384.0          0.979         0.506             0.742
MobileNetV3-Large     960.0          0.979           0.6             0.789
Baseline manual      1859.0          0.929         0.271               0.6
Acaso (referência)        -          0.924         0.071             0.498

Diferença da pureza balanceada (DINOv2 - outro), IC 95%:
  x MobileNetV3-Large  -0.047  [-0.088; -0.006]  -> DINOv2 perde
  x Baseline manual    +0.143  [+0.050; +0.228]  -> DINOv2 vence


## 5. Linhas de base: K-means sobre o DINOv2

K-means sobre as 9.325 fotos do universo (vista original, vetores
normalizados), com K = 50 (linha de base da H1) e K = 2 (exploratória), em 5
sementes. Avaliação nas 225 fotos da amostra aleatória, com o mapeamento por
maioria e as frações de "outros" calculados nos outros dois lotes (seções 1.5
e 1.6). Restrição: revocação de medidor >= 0,97 pelo limite inferior de 95%.

In [ ]:
#Célula 11
# K-means com K = 2 e K = 50 sobre o DINOv2, 5 sementes, mapeamento por lote deixado de fora
from training.cluster_eval import clustering_metrics, kmeans_clusters, leave_one_lot_out, one_hot

SEEDS = range(5)
R_MIN = 0.97
universe_original = dev_features[:, 0]                        # vista 0: a foto original
eval_rows = np.array([position[photos[i].pseudonym] for i in sample_idx])
eval_lots = np.array([photos[i].lot for i in sample_idx])

kmeans_runs, records = {}, []
start = time.perf_counter()
for n_clusters in (2, 50):
    for seed in SEEDS:
        clusters = kmeans_clusters(universe_original, n_clusters, seed)
        score, predicted = leave_one_lot_out(one_hot(clusters[eval_rows], n_clusters),
                                             is_outros, eval_lots)
        kmeans_runs[(n_clusters, seed)] = (score, predicted)
        records.append({"K": n_clusters, "semente": seed,
                        **clustering_metrics(score, predicted, is_outros)})
print(f"Tempo: {(time.perf_counter() - start) / 60:.1f} min\n")

kmeans_table = pd.DataFrame(records).set_index(["K", "semente"])
print(kmeans_table.round(3).to_string())

# Resumo por K: média e desvio entre sementes; quantas sementes cumprem a restrição
summary = kmeans_table.groupby("K").agg(["mean", "std"]).round(3)
print("\nMédia e desvio entre sementes:")
print(summary[["ap_outros", "revocacao_medidor", "revocacao_medidor_lim_inf",
               "revocacao_outros", "acuracia_balanceada"]].to_string())
complies = (kmeans_table["revocacao_medidor_lim_inf"] >= R_MIN).groupby("K").sum()
print(f"\nSementes que cumprem a restrição (lim. inf. >= {R_MIN}): {complies.to_dict()}")

Tempo: 0.7 min

            ap_outros  revocacao_medidor  revocacao_medidor_lim_inf  revocacao_outros  acuracia_balanceada  f1_macro  outros_previstos
K  semente                                                                                                                            
2  0            0.222              1.000                      0.986             0.000                0.500     0.480                 0
   1            0.227              1.000                      0.986             0.000                0.500     0.480                 0
   2            0.222              1.000                      0.986             0.000                0.500     0.480                 0
   3            0.222              1.000                      0.986             0.000                0.500     0.480                 0
   4            0.222              1.000                      0.986             0.000                0.500     0.480                 0
50 0            0.479              0.97

## 6. SCAN sobre o DINOv2

Vizinhos (k = 50, cosseno) na foto original das 9.325 fotos; agrupamento com
10 cabeças e auto-rotulagem, na configuração oficial para espinha congelada
(protocolo, seção 1.4), em 5 sementes. Primeiro o diagnóstico sem rótulos;
depois a avaliação nas 225 fotos, com o mesmo mapeamento por lote deixado de
fora do K-means.

In [7]:
#Célula 13
# SCAN em 5 sementes: diagnóstico sem rótulos e métricas com o mapeamento por lote deixado de fora
from training.scan import ScanConfig, mine_neighbors, run_scan

from dataclasses import replace

# Configuração original, antes da revisão de 20/10 (protocolo, seção 1.10): reportada no artigo
CFG = replace(ScanConfig(), unit_norm=False, min_occupied_fraction=0.0)
start = time.perf_counter()
neighbors = mine_neighbors(dev_features[:, 0], CFG.n_neighbors, device="cuda")
print(f"Vizinhos: {neighbors.shape} em {time.perf_counter() - start:.1f} s")

scan_runs, diagnostics, records = {}, [], []
for seed in SEEDS:
    start = time.perf_counter()
    probs, info = run_scan(dev_features, neighbors, seed, device="cuda", cfg=CFG)
    clusters = probs.argmax(axis=1)
    share = np.bincount(clusters, minlength=CFG.n_clusters) / len(clusters)
    head = info["scan"]["cabeca"]
    diagnostics.append({
        "semente": seed,
        "lr agrup.": info["scan"]["lr"],
        "lr autorrot.": info["autorrotulagem"]["lr"],
        "perda aval.": round(info["scan"]["perdas_avaliacao"][head], 4),
        "clusters ocupados": int((share > 0).sum()),
        "maior cluster (%)": round(100 * share.max(), 1),
        "confiantes (%)": round(100 * info["autorrotulagem"]["fracao_confiante"][-1], 1),
        "min": round((time.perf_counter() - start) / 60, 1),
    })
    score, predicted = leave_one_lot_out(probs[eval_rows], is_outros, eval_lots)
    scan_runs[seed] = (score, predicted, probs, info)
    records.append({"semente": seed, **clustering_metrics(score, predicted, is_outros)})

diag = pd.DataFrame(diagnostics).set_index("semente")
print("\nDiagnóstico sem rótulos:")
print(diag.to_string())
mapped_seed = int(diag["perda aval."].idxmin())
print(f"\nExecução para o mapeamento humano (menor perda de avaliação): semente {mapped_seed}")

scan_table = pd.DataFrame(records).set_index("semente")
print("\nMétricas (mapeamento por maioria, lote deixado de fora):")
print(scan_table.round(3).to_string())
print("\nMédia e desvio entre sementes:")
print(scan_table[["ap_outros", "revocacao_medidor", "revocacao_medidor_lim_inf",
                  "revocacao_outros", "acuracia_balanceada"]].agg(["mean", "std"]).round(3).to_string())
print(f"\nSementes viáveis (lim. inf. >= {R_MIN}): "
      f"{int((scan_table['revocacao_medidor_lim_inf'] >= R_MIN).sum())} de {len(SEEDS)}")

Vizinhos: (9325, 50) em 0.3 s

Diagnóstico sem rótulos:
         lr agrup.  lr autorrot.  perda aval.  clusters ocupados  maior cluster (%)  confiantes (%)  min
semente                                                                                                 
0              5.0          0.03       0.2082                  4               85.6           100.0  0.4
1              5.0          0.03       0.0759                  5               92.1           100.0  0.5
2              5.0          0.03       0.3168                  3               80.6           100.0  0.4
3              5.0          0.03       0.0724                  6               92.5           100.0  0.4
4              5.0          0.03       0.2698                  3               88.1           100.0  0.4

Execução para o mapeamento humano (menor perda de avaliação): semente 3

Métricas (mapeamento por maioria, lote deixado de fora):
         ap_outros  revocacao_medidor  revocacao_medidor_lim_inf  revocacao_ou

In [8]:
#Célula 14
# Diagnóstico sem rótulos do quase-colapso: escala das características e fase em que ocorre
from training.scan import train_scan

norms = np.linalg.norm(dev_features[:, 0], axis=1)
print(f"Norma dos vetores DINOv2: média {norms.mean():.1f} | mín {norms.min():.1f} | máx {norms.max():.1f}")

rows = []
for seed in SEEDS:
    head, info_scan = train_scan(dev_features, neighbors, CFG, seed, device="cuda")
    with torch.no_grad():
        logits = head(torch.as_tensor(dev_features[:, 0], device="cuda"))
    p = logits.softmax(dim=1).cpu().numpy()
    share_scan = np.bincount(p.argmax(1), minlength=CFG.n_clusters) / len(p)
    share_final = np.bincount(scan_runs[seed][2].argmax(1), minlength=CFG.n_clusters) / len(p)
    rows.append({
        "semente": seed,
        "logit: desvio": round(logits.std().item(), 1),
        "p máx. média": round(p.max(1).mean(), 3),
        "perda 1ª época": round(info_scan["historico"][0], 2),
        "perda última": round(info_scan["historico"][-1], 2),
        "ocupados após agrup.": int((share_scan > 0).sum()),
        "maior após agrup. (%)": round(100 * share_scan.max(), 1),
        "ocupados no final": int((share_final > 0).sum()),
    })
print(pd.DataFrame(rows).set_index("semente").to_string())

Norma dos vetores DINOv2: média 46.8 | mín 40.3 | máx 50.7
         logit: desvio  p máx. média  perda 1ª época  perda última  ocupados após agrup.  maior após agrup. (%)  ocupados no final
semente                                                                                                                           
0               3875.9           1.0           -2.29         -2.04                     4                   85.6                  4
1               6386.7           1.0           -2.17         -2.35                     5                   92.1                  5
2               6632.6           1.0           -1.99         -2.09                     3                   80.6                  3
3               2840.6           1.0           -2.42         -1.93                     6                   92.5                  6
4               5386.9           1.0           -1.87         -2.27                     3                   88.1                  3


## 7. SCAN corrigido (revisão de 20/10)

Mesma configuração, com os vetores normalizados antes da cabeça e a detecção
do quase-colapso (protocolo, seção 1.10). Revisão posterior aos resultados com
rótulos da seção 6, motivada só pelo diagnóstico sem rótulos da célula 14. É
esta execução que decide a H1.

In [9]:
#Célula 16
# SCAN corrigido em 5 sementes: diagnóstico sem rótulos e métricas
CFG_FIXED = ScanConfig()   # unit_norm=True, min_occupied_fraction=0.5
assert CFG_FIXED.unit_norm and CFG_FIXED.min_occupied_fraction == 0.5

scan_fixed, diagnostics, records = {}, [], []
for seed in SEEDS:
    start = time.perf_counter()
    probs, info = run_scan(dev_features, neighbors, seed, device="cuda", cfg=CFG_FIXED)
    share = np.bincount(probs.argmax(axis=1), minlength=CFG_FIXED.n_clusters) / len(probs)
    head = info["scan"]["cabeca"]
    diagnostics.append({
        "semente": seed,
        "lr agrup.": info["scan"]["lr"],
        "lr autorrot.": info["autorrotulagem"]["lr"],
        "perda aval.": round(info["scan"]["perdas_avaliacao"][head], 4),
        "clusters ocupados": int((share > 0).sum()),
        "maior cluster (%)": round(100 * share.max(), 1),
        "confiantes (%)": round(100 * info["autorrotulagem"]["fracao_confiante"][-1], 1),
        "min": round((time.perf_counter() - start) / 60, 1),
    })
    score, predicted = leave_one_lot_out(probs[eval_rows], is_outros, eval_lots)
    scan_fixed[seed] = (score, predicted, probs, info)
    records.append({"semente": seed, **clustering_metrics(score, predicted, is_outros)})

diag_fixed = pd.DataFrame(diagnostics).set_index("semente")
print("Diagnóstico sem rótulos:")
print(diag_fixed.to_string())
mapped_seed = int(diag_fixed["perda aval."].idxmin())
print(f"\nExecução para o mapeamento humano (menor perda de avaliação): semente {mapped_seed}")

scan_fixed_table = pd.DataFrame(records).set_index("semente")
print("\nMétricas (mapeamento por maioria, lote deixado de fora):")
print(scan_fixed_table.round(3).to_string())
print("\nMédia e desvio entre sementes:")
print(scan_fixed_table[["ap_outros", "revocacao_medidor", "revocacao_medidor_lim_inf",
                        "revocacao_outros", "acuracia_balanceada"]].agg(["mean", "std"]).round(3).to_string())
print(f"\nSementes viáveis (lim. inf. >= {R_MIN}): "
      f"{int((scan_fixed_table['revocacao_medidor_lim_inf'] >= R_MIN).sum())} de {len(SEEDS)}")

Diagnóstico sem rótulos:
         lr agrup.  lr autorrot.  perda aval.  clusters ocupados  maior cluster (%)  confiantes (%)  min
semente                                                                                                 
0              5.0          0.03      -2.6483                 50                3.2             5.7  0.4
1              5.0          0.03      -2.6515                 50                3.9             5.3  0.3
2              5.0          0.03      -2.6450                 50                3.4             5.0  0.5
3              5.0          0.03      -2.6348                 50                3.2             6.8  0.4
4              5.0          0.03      -2.6413                 50                3.7             6.2  0.4

Execução para o mapeamento humano (menor perda de avaliação): semente 1

Métricas (mapeamento por maioria, lote deixado de fora):
         ap_outros  revocacao_medidor  revocacao_medidor_lim_inf  revocacao_outros  acuracia_balanceada  f1_m

## 8. H1: SCAN × K-means e viabilidade

Decisão da H1 (protocolo, seção 1.7): diferença da AP média de "outros" entre
as 5 sementes, SCAN corrigido contra K-means (K = 50), por bootstrap pareado e
estratificado por classe, com 2.000 reamostragens. A comparação do SCAN
original é só descritiva. A viabilidade (seção 1.6) é reportada à parte.

In [11]:
# Célula 18

# H1: SCAN corrigido x K-means (K = 50); SCAN original só descritivo; viabilidade à parte
import importlib

import training.cluster_eval as cluster_eval
importlib.reload(cluster_eval)   # o módulo ganhou funções depois da importação

configs = {
    "SCAN corrigido": [scan_fixed[s] for s in SEEDS],
    "SCAN original": [scan_runs[s] for s in SEEDS],
    "K-means K = 50": [kmeans_runs[(50, s)] for s in SEEDS],
    "K-means K = 2": [kmeans_runs[(2, s)] for s in SEEDS],
}
scores = {name: np.stack([run[0] for run in runs]) for name, runs in configs.items()}

print("Diferença da AP média de 'outros' (A - K-means K = 50), IC 95%:")
for name, role in (("SCAN corrigido", "H1, confirmatória"), ("SCAN original", "descritiva")):
    point, low, high = cluster_eval.paired_bootstrap_mean_ap(
        scores[name], scores["K-means K = 50"], is_outros)
    verdict = "SCAN vence" if low > 0 else ("SCAN perde" if high < 0 else "empate")
    print(f"  {name:15} {point:+.3f}  [{low:+.3f}; {high:+.3f}]  -> {verdict}  ({role})")

# Viabilidade por configuração: medidores perdidos e limite inferior, por semente
meters = int((is_outros == 0).sum())
rows = {}
for name, runs in configs.items():
    lost = [int(run[1][is_outros == 0].sum()) for run in runs]
    lower = [cluster_eval.meter_recall_lower(run[1], is_outros) for run in runs]
    rows[name] = {
        "AP média": round(cluster_eval.mean_ap(scores[name], is_outros), 3),
        "medidores perdidos (por semente)": lost,
        "lim. inf. mín.": round(min(lower), 3),
        "lim. inf. máx.": round(max(lower), 3),
        "viáveis": f"{sum(v >= R_MIN for v in lower)} de {len(runs)}",
    }
print(f"\nViabilidade (revocação de medidor >= {R_MIN} pelo limite inferior; {meters} medidores):")
print(pd.DataFrame(rows).T.to_string())

Diferença da AP média de 'outros' (A - K-means K = 50), IC 95%:
  SCAN corrigido  +0.228  [+0.111; +0.303]  -> SCAN vence  (H1, confirmatória)
  SCAN original   -0.028  [-0.170; +0.075]  -> empate  (descritiva)

Viabilidade (revocação de medidor >= 0.97 pelo limite inferior; 208 medidores):
               AP média medidores perdidos (por semente) lim. inf. mín. lim. inf. máx. viáveis
SCAN corrigido    0.683                  [8, 6, 6, 9, 4]          0.926          0.957  0 de 5
SCAN original     0.426                 [0, 7, 0, 6, 12]          0.908          0.986  2 de 5
K-means K = 50    0.454                  [5, 8, 6, 6, 7]          0.932           0.95  0 de 5
K-means K = 2     0.223                  [0, 0, 0, 0, 0]          0.986          0.986  5 de 5
